In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_gold.dev_sh.customers (
    customer_id INT,
    name STRING,
    city STRING,
    signup_date DATE
);

INSERT INTO dev_gold.dev_sh.customers VALUES
  (1, 'Alice', 'New York', DATE('2022-01-01')),
  (2, 'Bob', 'Los Angeles', DATE('2022-03-15')),
  (3, 'Carol', 'Chicago', DATE('2023-06-10')),
  (4, 'Dave', 'Houston', DATE('2024-02-20')),
  (5, 'Eve', 'Phoenix', DATE('2023-12-01'));


CREATE TABLE IF NOT EXISTS dev_gold.dev_sh.products (
    product_id INT,
    product_name STRING,
    price DOUBLE
);

INSERT INTO dev_gold.dev_sh.products VALUES
  (101, 'Laptop', 1200.00),
  (102, 'Mouse', 25.00),
  (103, 'Keyboard', 55.00),
  (104, 'Monitor', 300.00);



CREATE TABLE IF NOT EXISTS dev_gold.dev_sh.orders (
    order_id INT,
    customer_id INT,
    order_date DATE,
    payment_method STRING
);

INSERT INTO dev_gold.dev_sh.orders VALUES
  (1001, 1, DATE('2024-07-01'), 'Credit Card'),
  (1002, 1, DATE('2024-07-15'), 'PayPal'),
  (1003, 2, DATE('2024-07-10'), 'Credit Card'),
  (1004, 3, DATE('2024-08-01'), 'Debit Card'),
  (1005, 4, DATE('2024-08-01'), 'UPI');



CREATE TABLE IF NOT EXISTS dev_gold.dev_sh.order_items (
    order_id INT,
    product_id INT,
    quantity INT
);

INSERT INTO dev_gold.dev_sh.order_items VALUES
  (1001, 101, 1),
  (1001, 102, 2),
  (1002, 103, 1),
  (1003, 104, 1),
  (1004, 102, 3),
  (1005, 101, 1),
  (1005, 104, 1);


In [0]:
1. Total number of orders per customer
2.Total sales amount per customer (
3. Find customers who placed more than 1 order
4.List all products that have never been ordere
5. Show top 2 cities by number of customers
6.  Which customer spent the most?
7. Find the most frequently ordered product
8.  List all orders placed in August 2024
9.  Show all order details including:




10. 📈 Average order value per payment method

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_gold.dev_sh.transactions (
    transaction_id BIGINT,
    user_id        INT,
    product_id     INT,
    quantity       INT,
    price          DOUBLE,
    transaction_date DATE,
    payment_method STRING
) USING DELTA;


INSERT INTO dev_gold.dev_sh.transactions VALUES
  (1001, 1, 101, 2, 19.99, DATE('2025-08-01'), 'Credit Card'),
  (1002, 2, 102, 1, 49.99, DATE('2025-08-01'), 'PayPal'),
  (1003, 1, 103, 3, 15.00, DATE('2025-08-02'), 'Debit Card'),
  (1004, 3, 104, 2, 99.00, DATE('2025-08-03'), 'Credit Card'),
  (1005, 4, 101, 1, 19.99, DATE('2025-08-03'), 'UPI'),
  (1006, 2, 105, 1, 25.00, DATE('2025-08-04'), 'Credit Card');


In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema","")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(catalog)
print(schema)

path = (f"{catalog}.{schema}.")
print(path)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_gold.dev_sh.transactions (
    transaction_id BIGINT,
    user_id        INT,
    product_id     INT,
    quantity       INT,
    price          DOUBLE,
    transaction_date DATE,
    payment_method STRING
) USING DELTA;


INSERT INTO dev_gold.dev_sh.transactions VALUES
  (1001, 1, 101, 2, 19.99, DATE('2025-08-01'), 'Credit Card'),
  (1002, 2, 102, 1, 49.99, DATE('2025-08-01'), 'PayPal'),
  (1003, 1, 103, 3, 15.00, DATE('2025-08-02'), 'Debit Card'),
  (1004, 3, 104, 2, 99.00, DATE('2025-08-03'), 'Credit Card'),
  (1005, 4, 101, 1, 19.99, DATE('2025-08-03'), 'UPI'),
  (1006, 2, 105, 1, 25.00, DATE('2025-08-04'), 'Credit Card');


In [0]:
from pyspark.sql.functions import expr, rand, round
from pyspark.sql.types import *
import datetime

n = 100000

df = spark.range(n).withColumn("transaction_id", expr("id + 1000")) \
    .withColumn("user_id", (rand() * 1000).cast("int")) \
    .withColumn("product_id", (rand() * 50 + 100).cast("int")) \
    .withColumn("quantity", round(rand() * 5 + 1, 0)) \
    .withColumn("price", round(rand() * 100 + 10, 2)) \
    .withColumn("order_date", expr("date_sub(current_date(), cast(rand() * 30 as int))")) \
    .drop("id")

table_location = "dev_gold."+"dev_sh.""retail_sales"
print(table_location)
df.write.mode("overwrite").saveAsTable(table_location )    


row_table = spark.read.table(table_location)
display(row_table).limit(10)



In [0]:
row_table.count()

In [0]:
from pyspark.sql.functions import col

stage_df = row_table \
    .withColumn("total_amount", col("quantity") * col("price")) \
    .dropDuplicates(["transaction_id"]) \
    .na.drop(subset=["user_id", "product_id", "price"]) \
    .filter(col("quantity") > 0) \
    # .cache()


In [0]:
stage_df.count()
stage_df.display(10)

In [0]:
%sql
DROP CATALOG IF EXISTS workspace CASCADE;
DROP CATALOG IF EXISTS prod_sh CASCADE;
DROP CATALOG IF EXISTS prod_sh CASCADE;